In [ ]:
import redis

HOSTNAME = "ENDPOINT"
PORT = 10004
PASSWORD = "PASSWORD"

r = redis.Redis(
    host=HOSTNAME,
    port=PORT,
    password=PASSWORD,
    decode_responses=True
)

r.ping()  # Should return True

In [ ]:
r.decr()

In [ ]:
import random
from datetime import datetime, timedelta

try:
    from faker import Faker
    USE_FAKER = True
except ImportError:
    print("La bibliothèque 'faker' n'est pas installée. Utilisation de données aléatoires basiques.")
    USE_FAKER = False

def generate_navigation_history(user_id, num_entries=50):
    """Génère un historique de navigation fictif pour un utilisateur."""
    pages = ["home", "product/101", "product/202", "product/303", "cart", "checkout", "category/electronics", "category/books"]

    history = []
    for _ in range(num_entries):
        page = random.choice(pages)
        if USE_FAKER:
            fake = Faker()
            timestamp = fake.date_time_this_year()
        else:
            # Alternative sans Faker : timestamp aléatoire dans les 60 derniers jours
            timestamp = datetime.now() - timedelta(days=random.randint(0, 60), hours=random.randint(0, 23), minutes=random.randint(0, 59))

        history.append({
            "user_id": user_id,
            "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            "page": page
        })
    return history

# Générer des données pour plusieurs utilisateurs
num_users = 10
num_entries_per_user = 50

navigation_data = []
for i in range(num_users):
    user_id = f"user_{i+1:03d}"  # Format: user_001, user_002, etc.
    user_history = generate_navigation_history(user_id, num_entries=num_entries_per_user)
    navigation_data.extend(user_history)

# Afficher un résumé
print(f"Données générées pour {num_users} utilisateurs")
print(f"Total d'entrées de navigation : {len(navigation_data)}")
print(f"\nAperçu des 10 premières entrées :")
for entry in navigation_data[:10]:
    print(f"{entry['user_id']} - {entry['timestamp']} - {entry['page']}")


In [ ]:
import pandas as pd

df = pd.DataFrame(navigation_data)
df.to_dict()


In [ ]:
from datetime import datetime

print("🔄 Nettoyage de Redis avant la démo...")
r.flushdb()  # Nettoie la base pour la démo

print("\n📥 Stockage des données de navigation dans Redis...\n")

# 1. Stocker l'historique de navigation avec Sorted Sets (timestamp = score)
for entry in navigation_data:
    user_id = entry['user_id']
    page = entry['page']
    timestamp = entry['timestamp']
    
    # Convertir le timestamp en score Unix
    dt = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
    score = dt.timestamp()
    
    # Clé: nav:user_id - Valeur: page avec timestamp comme score
    r.zadd(f"nav:{user_id}", {f"{timestamp}|{page}": score})
    
    # Compter les visites par page pour chaque utilisateur (Hash)
    r.hincrby(f"page_count:{user_id}", page, 1)
    
    # Ajouter la page aux pages uniques visitées (Set)
    r.sadd(f"unique_pages:{user_id}", page)
    
    # Statistiques globales : compter les visites totales par page
    r.zincrby("global_page_stats", 1, page)

print(f"✅ {len(navigation_data)} entrées stockées dans Redis!\n")

# 2. REQUÊTES ET ANALYSES
print("=" * 60)
print("📊 ANALYSES ET REQUÊTES")
print("=" * 60)

# Exemple : Historique d'un utilisateur
user_example = "user_001"
print(f"\n1️⃣  Historique récent de {user_example} (10 dernières visites) :")
recent_nav = r.zrevrange(f"nav:{user_example}", 0, 9)
for i, item in enumerate(recent_nav, 1):
    timestamp, page = item.split('|')
    print(f"   {i}. {timestamp} - {page}")

# Pages visitées par utilisateur
print(f"\n2️⃣  Pages visitées par {user_example} :")
page_counts = r.hgetall(f"page_count:{user_example}")
for page, count in sorted(page_counts.items(), key=lambda x: int(x[1]), reverse=True)[:5]:
    print(f"   {page}: {count} visites")

# Pages uniques visitées
unique_count = r.scard(f"unique_pages:{user_example}")
print(f"\n3️⃣  Nombre de pages uniques visitées par {user_example}: {unique_count}")

# Top 5 pages globales les plus visitées
print(f"\n4️⃣  Top 5 des pages les plus visitées (tous utilisateurs) :")
top_pages = r.zrevrange("global_page_stats", 0, 4, withscores=True)
for i, (page, score) in enumerate(top_pages, 1):
    print(f"   {i}. {page}: {int(score)} visites")

# Statistiques par utilisateur
print(f"\n5️⃣  Statistiques par utilisateur :")
for i in range(1, 6):  # Afficher 5 premiers utilisateurs
    user_id = f"user_{i:03d}"
    total_visits = r.zcard(f"nav:{user_id}")
    unique_pages = r.scard(f"unique_pages:{user_id}")
    print(f"   {user_id}: {total_visits} visites, {unique_pages} pages uniques")

# Navigation dans une période spécifique
print(f"\n6️⃣  Activité de {user_example} durant les 30 derniers jours :")
thirty_days_ago = (datetime.now() - timedelta(days=30)).timestamp()
now = datetime.now().timestamp()
recent_activity = r.zcount(f"nav:{user_example}", thirty_days_ago, now)
print(f"   {recent_activity} visites dans les 30 derniers jours")

print("\n" + "=" * 60)
print("✨ Démo Redis terminée!")
print("=" * 60)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 📊 GRAPHIQUE 1 : Top pages visitées (toutes utilisateurs)
print("📊 Génération des graphiques...\n")

top_pages_data = r.zrevrange("global_page_stats", 0, -1, withscores=True)
pages_df = pd.DataFrame(top_pages_data, columns=['page', 'visites'])
pages_df['visites'] = pages_df['visites'].astype(int)

fig1 = px.bar(pages_df, 
              x='page', 
              y='visites',
              title='📈 Top des pages les plus visitées (tous utilisateurs)',
              labels={'page': 'Page', 'visites': 'Nombre de visites'},
              color='visites',
              color_continuous_scale='Blues')
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

# 📊 GRAPHIQUE 2 : Distribution des pages (Pie Chart)
fig2 = px.pie(pages_df, 
              values='visites', 
              names='page',
              title='🥧 Répartition des visites par page',
              hole=0.3)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

# 📊 GRAPHIQUE 3 : Statistiques par utilisateur
users_stats = []
for i in range(1, num_users + 1):
    user_id = f"user_{i:03d}"
    total_visits = r.zcard(f"nav:{user_id}")
    unique_pages = r.scard(f"unique_pages:{user_id}")
    users_stats.append({
        'user_id': user_id,
        'total_visites': total_visits,
        'pages_uniques': unique_pages
    })

users_df = pd.DataFrame(users_stats)

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Total des visites par utilisateur', 'Pages uniques visitées par utilisateur')
)

fig3.add_trace(
    go.Bar(x=users_df['user_id'], y=users_df['total_visites'], name='Total visites', marker_color='indianred'),
    row=1, col=1
)

fig3.add_trace(
    go.Bar(x=users_df['user_id'], y=users_df['pages_uniques'], name='Pages uniques', marker_color='lightsalmon'),
    row=1, col=2
)

fig3.update_layout(height=400, title_text="👥 Statistiques par utilisateur", showlegend=False)
fig3.update_xaxes(tickangle=-45)
fig3.show()

# 📊 GRAPHIQUE 4 : Heatmap des pages visitées par utilisateur
heatmap_data = []
pages_list = pages_df['page'].tolist()

for user_id in users_df['user_id']:
    user_data = r.hgetall(f"page_count:{user_id}")
    row = [int(user_data.get(page, 0)) for page in pages_list]
    heatmap_data.append(row)

fig4 = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=pages_list,
    y=users_df['user_id'].tolist(),
    colorscale='Viridis',
    text=heatmap_data,
    texttemplate='%{text}',
    textfont={"size": 10}
))

fig4.update_layout(
    title='🔥 Heatmap : Visites par utilisateur et par page',
    xaxis_title='Pages',
    yaxis_title='Utilisateurs',
    height=500
)
fig4.update_xaxes(tickangle=-45)
fig4.show()

# 📊 GRAPHIQUE 5 : Timeline d'activité pour un utilisateur spécifique
user_timeline = "user_001"
nav_history = r.zrange(f"nav:{user_timeline}", 0, -1, withscores=True)

timeline_data = []
for item, score in nav_history:
    timestamp_str, page = item.split('|')
    timeline_data.append({
        'timestamp': datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S"),
        'page': page
    })

timeline_df = pd.DataFrame(timeline_data)
timeline_df = timeline_df.sort_values('timestamp')

# Créer un compteur cumulatif de visites
timeline_df['visite_num'] = range(1, len(timeline_df) + 1)

fig5 = px.scatter(timeline_df, 
                  x='timestamp', 
                  y='page',
                  color='page',
                  size_max=10,
                  title=f'📅 Timeline de navigation pour {user_timeline}',
                  labels={'timestamp': 'Date et heure', 'page': 'Page visitée'})
fig5.update_traces(marker=dict(size=8))
fig5.show()

# 📊 GRAPHIQUE 6 : Comparaison Visites totales vs Pages uniques
fig6 = px.scatter(users_df, 
                  x='pages_uniques', 
                  y='total_visites',
                  text='user_id',
                  title='📊 Relation : Pages uniques vs Total des visites',
                  labels={'pages_uniques': 'Pages uniques visitées', 'total_visites': 'Total des visites'},
                  size='total_visites',
                  color='total_visites',
                  color_continuous_scale='thermal')
fig6.update_traces(textposition='top center')
fig6.show()

print("\n✅ Tous les graphiques ont été générés avec succès!")